# SVD & Low-Rank Approximation

Wiki reference for [SVD & low-rank approximation](https://ml-viz-ruby.vercel.app/wiki/svd-low-rank).

**The idea in one sentence.** The SVD writes any matrix as $\sum_i \sigma_i u_i v_i^\top$ with
**decreasing** singular values, and the **Eckart-Young theorem** says truncating to the top $k$
terms is the *best possible* rank-$k$ approximation — with error exactly
$\sqrt{\sum_{i>k}\sigma_i^2}$ — the basis of compression, denoising, and PCA.

We compute SVD truncations from scratch, **validate the Eckart-Young error and singular-value
ordering**, then cover the gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('dark_background')
plt.rcParams.update({
    'axes.facecolor': '#1a1d27',
    'figure.facecolor': '#0f1117',
    'axes.edgecolor': '#3a3d4a',
    'grid.color': '#2a2d3a',
    'text.color': '#e2e8f0',
    'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'font.size': 11,
})

## 1 · Eckart-Young bound: actual error = discarded tail

In [ ]:
rng = np.random.default_rng(0)
A = rng.normal(size=(60, 40)) @ rng.normal(size=(40, 40))  # full-rank 60×40

U, s, Vt = np.linalg.svd(A, full_matrices=False)

print(f"A shape: {A.shape}, rank: {np.linalg.matrix_rank(A)}")
print(f"Largest 10 singular values: {s[:10].round(2)}")
print()
print(f"{'k':>4}  {'actual ‖A-Aₖ‖_F':>18}  {'theory √(Σ σᵢ²)':>18}  {'match':>6}")
print("-" * 55)

for k in [1, 5, 10, 20, 30, 39]:
    A_k = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
    actual  = np.linalg.norm(A - A_k, 'fro')
    theory  = np.sqrt((s[k:] ** 2).sum())
    print(f"{k:>4}  {actual:>18.8f}  {theory:>18.8f}  {np.isclose(actual, theory)!s:>6}")

### Validate: Eckart-Young — the truncation error is the tail of singular values

The rank-$k$ SVD truncation's Frobenius error equals $\sqrt{\sum_{i>k}\sigma_i^2}$ (the dropped
singular values), and the singular values come out in decreasing order. We confirm both.

In [ ]:
for k in [1, 5, 20]:
    A_k = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
    actual = np.linalg.norm(A - A_k, 'fro')
    theory = np.sqrt((s[k:] ** 2).sum())
    assert np.isclose(actual, theory), 'Eckart-Young: rank-k error = sqrt(sum of dropped singular values squared)'
assert all(s[i] >= s[i+1] - 1e-9 for i in range(len(s) - 1)), 'singular values are sorted decreasing'
print(f'Eckart-Young verified; largest singular values: {s[:5].round(1)}')
print('\n✅ SVD truncation is the provably-best low-rank approximation')

## 2 · SVD image compression

We generate a synthetic grayscale image (or use numpy to create a structured one) and compress it by keeping only the top-k singular values.

In [ ]:
# Build a structured synthetic image (checkerboard + smooth gradient)
n = 128
x_idx = np.arange(n)
xx, yy = np.meshgrid(x_idx, x_idx)

# Low-rank signal: sum of a few outer products
rng2 = np.random.default_rng(42)
rank_true = 15
U_img = rng2.normal(size=(n, rank_true))
V_img = rng2.normal(size=(n, rank_true))
S_img = np.exp(-np.arange(rank_true) * 0.3) * 100  # decaying singular values
img_clean = U_img @ np.diag(S_img) @ V_img.T

# Normalise to [0, 1]
img_clean = (img_clean - img_clean.min()) / (img_clean.max() - img_clean.min())
# Add mild noise
img_noisy = img_clean + 0.05 * rng2.normal(size=(n, n))
img_noisy = np.clip(img_noisy, 0, 1)

# SVD of the noisy image
U_s, s_img, Vt_s = np.linalg.svd(img_noisy, full_matrices=False)

k_values = [1, 5, 10, 20, 50]
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.ravel()

axes[0].imshow(img_noisy, cmap='gray', vmin=0, vmax=1)
axes[0].set_title(f'Original (noisy), {n}×{n}')
axes[0].axis('off')

errors = []
for ax, k in zip(axes[1:], k_values):
    recon = U_s[:, :k] @ np.diag(s_img[:k]) @ Vt_s[:k, :]
    recon = np.clip(recon, 0, 1)
    err = np.linalg.norm(img_noisy - recon, 'fro') / np.linalg.norm(img_noisy, 'fro')
    errors.append((k, err))
    n_params = k * (n + n + 1)
    compress = n * n / n_params
    ax.imshow(recon, cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'k={k}  rel.err={err:.3f}  {compress:.1f}× compression')
    ax.axis('off')

plt.suptitle('SVD Image Compression', y=1.01)
plt.tight_layout()
plt.show()

## 3 · Reconstruction error and scree plot

In [ ]:
k_range = np.arange(1, 60)
rel_errors = []
for k in k_range:
    theory_err = np.sqrt((s_img[k:] ** 2).sum())
    rel_errors.append(theory_err / np.linalg.norm(img_noisy, 'fro'))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].semilogy(k_range, rel_errors, color='#6366f1', lw=2)
axes[0].axvline(rank_true, color='#f59e0b', ls='--', label=f'True rank = {rank_true}')
axes[0].set(xlabel='k (rank)', ylabel='Relative Frobenius error',
            title='Reconstruction error vs rank')
axes[0].legend()
axes[0].grid(True, alpha=0.3, which='both')

axes[1].plot(s_img[:50], 'o-', color='#34d399', ms=4, lw=1.5)
axes[1].axvline(rank_true - 1, color='#f59e0b', ls='--', label=f'Elbow at k≈{rank_true}')
axes[1].set(xlabel='Index i', ylabel='σᵢ', title='Scree plot (singular values)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **choosing k** | too small loses signal, too large keeps noise — use the energy/scree curve |
| **full-rank data** | little compression if singular values don't decay |
| **cost** | full SVD is $O(mn\min(m,n))$ — use truncated/randomized SVD for large matrices |
| **not centering** | for PCA, center the data first |
| **sign ambiguity** | singular vectors are defined up to sign |

Demo: a few singular values capture 95% of a low-rank signal's energy.

In [ ]:
# Why SVD is the workhorse of compression/denoising: for a LOW-RANK signal, almost all the
# 'energy' (sum of squared singular values) lives in the top few components — so a handful of
# rank-1 terms reconstruct it, and the rest (mostly noise) can be dropped. We show how few
# components capture 95% of the energy of the structured image.
energy = np.cumsum(s_img ** 2) / np.sum(s_img ** 2)
k95 = int(np.argmax(energy >= 0.95) + 1)
print(f'{k95} of {len(s_img)} singular values capture 95% of the image energy (true rank ~ {rank_true})')
assert k95 <= 3 * rank_true, 'a low-rank signal concentrates its energy in the top singular values'
print('\nEnergy concentrates in the top singular values -> keep those, drop the rest = compression + denoising.')

---

## ✏️ Your turn

### Exercise 1 — SVD from scratch (rank-1 updates)

Implement the rank-k approximation from scratch using the sum-of-layers formula, without using matrix multiplication shorthand.

In [ ]:
def rank_k_approx(U, s, Vt, k):
    """Build Aₖ as sum of k rank-1 outer products: Σᵢ σᵢ uᵢ vᵢᵀ"""
    # TODO(you): implement as an explicit loop over i = 0..k-1
    pass

# Test on our matrix A
k = 10
A_k_ref = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
# A_k_yours = rank_k_approx(U, s, Vt, k)
# assert np.allclose(A_k_ref, A_k_yours), "Mismatch!"
print("Implement rank_k_approx and uncomment the assert.")

### Exercise 2 — verify the shrinkage formula for Ridge

Ridge regression shrinks singular values: the SVD of $(X^\top X + \lambda I)^{-1} X^\top$ equals
$V \,\text{diag}\!\left(\frac{\sigma_i}{\sigma_i^2 + \lambda}\right)\! U^\top$.
Verify this numerically.

In [ ]:
rng3 = np.random.default_rng(5)
X = rng3.normal(size=(30, 10))  # n=30 samples, d=10 features

# TODO(you):
# 1. Compute SVD of X: U_x, s_x, Vt_x = np.linalg.svd(X, full_matrices=False)
# 2. Compute Ridge pseudo-inverse directly: inv((X.T @ X) + lam*I) @ X.T
# 3. Compute Ridge pseudo-inverse via SVD formula: V @ diag(s_x/(s_x**2+lam)) @ U_x.T
# 4. Assert they're equal

lam = 0.5
print("Implement and verify the Ridge SVD shrinkage formula.")

<details>
<summary>Solutions</summary>

```python
# Exercise 1
def rank_k_approx(U, s, Vt, k):
    result = np.zeros((U.shape[0], Vt.shape[1]))
    for i in range(k):
        result += s[i] * np.outer(U[:, i], Vt[i, :])
    return result

assert np.allclose(A_k_ref, rank_k_approx(U, s, Vt, 10))
print("Exercise 1: pass!")

# Exercise 2
U_x, s_x, Vt_x = np.linalg.svd(X, full_matrices=False)
ridge_direct = np.linalg.inv(X.T @ X + lam * np.eye(X.shape[1])) @ X.T
ridge_svd    = Vt_x.T @ np.diag(s_x / (s_x**2 + lam)) @ U_x.T
assert np.allclose(ridge_direct, ridge_svd, atol=1e-10)
print("Exercise 2: pass!")
```
</details>

## Key takeaways

- **SVD = $\sum_i \sigma_i u_i v_i^\top$** with decreasing singular values (verified).
- **Eckart-Young:** the top-$k$ truncation is the best rank-$k$ approximation, error
  $\sqrt{\sum_{i>k}\sigma_i^2}$ (verified).
- **Energy concentrates** in the top components for low-rank signals (demo) — compression &
  denoising.
- **Foundation of PCA, recommenders, and matrix completion.**